# NB06: 공공시설 (유휴공간 후보) 좌표화 및 통합

**목적**: 드론 스테이션 후보 부지로 활용 가능한 공공시설물을 GeoDataFrame으로 통합

**입력**:
- 주차장 정보 CSV (`서울시 공영주차장 안내 정보.csv`)

**출력**: `processed/public_facilities.gpkg`

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

BASE = Path(r"E:\서울시데이터경진대회\Aero-Logic-Seoul")
RAW = BASE / "00_data"
OUT = BASE / "processed"
OUT.mkdir(exist_ok=True)

## 1. 주차장 데이터 로드 및 전처리

In [4]:
import pandas as pd
import geopandas as gpd
import requests
from sklearn.preprocessing import MinMaxScaler
import urllib3

# SSL 경고 숨기기 (이전에 겪으셨던 인증서 에러 방지용)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# V-World API 키 설정 (본인 키로 변경 필요)
VWORLD_KEY = "54292B3F-F5D3-32BB-A331-AE638BDB0C1D"

def get_coordinates_vworld(address):
    """주소를 받아 V-World API로 위도(y), 경도(x)를 반환하는 함수"""
    url = "https://api.vworld.kr/req/address"
    params = {
        "service": "address",
        "request": "getcoord",
        "version": "2.0",
        "crs": "epsg:4326",
        "address": address,
        "format": "json",
        "type": "ROAD", # 도로명 기준 (안 나오면 PARCEL(지번)로 재시도 가능)
        "key": VWORLD_KEY
    }
    
    try:
        # 이전에 발생했던 SSL 에러를 막기 위해 verify=False 필수
        response = requests.get(url, params=params, verify=False, timeout=5)
        result = response.json()
        
        if result['response']['status'] == 'OK':
            x = result['response']['result']['point']['x'] # 경도
            y = result['response']['result']['point']['y'] # 위도
            return float(y), float(x)
    except Exception as e:
        pass
        
    return None, None

# ==========================================
# 1. 주차장 데이터 로드 및 분리
# ==========================================
pk = pd.read_csv(RAW / "서울시 공영주차장 안내 정보.csv", encoding="cp949")
print(f"주차장 원본: {len(pk)}개소")

# 이미 좌표가 있는 정상 데이터와 없는 데이터(결측치) 분리
pk_valid = pk.dropna(subset=['위도', '경도']).copy()
pk_missing = pk[pk['위도'].isna() | pk['경도'].isna()].copy()

print(f"좌표 정상: {len(pk_valid)}개소 | 좌표 누락: {len(pk_missing)}개소 (복구 시도)")

# ==========================================
# 2. 누락된 데이터 V-World API로 지오코딩 진행
# ==========================================
print("🌐 누락된 주소 좌표 복구 중... (약간의 시간이 소요됩니다)")

# apply를 사용해 각 행의 주소값을 함수에 넣고, 반환된 위/경도를 새로운 컬럼에 할당
# tqdm을 설치하셨다면 apply 대신 progress_apply를 쓰면 진행률 바를 볼 수 있습니다.
pk_missing['위도'], pk_missing['경도'] = zip(*pk_missing['주소'].apply(get_coordinates_vworld))

# API로도 못 찾은 찐 결측치 최종 제거
pk_recovered = pk_missing.dropna(subset=['위도', '경도']).copy()
print(f"복구 성공: {len(pk_recovered)}개소")

# ==========================================
# 3. 데이터 병합 및 GeoDataFrame 생성
# ==========================================
# 기존 정상 데이터와 복구된 데이터를 위아래로 합치기
pk_final = pd.concat([pk_valid, pk_recovered], ignore_index=True)
print(f"최종 확보된 주차장 데이터: {len(pk_final)}개소")

# GeoDataFrame 변환
gdf_parking = gpd.GeoDataFrame(
    pk_final,
    geometry=gpd.points_from_xy(pk_final["경도"], pk_final["위도"]),
    crs="EPSG:4326"
)

# 핵심 컬럼만 추출
gdf_parking = gdf_parking[[
    "주차장코드", "주차장명", "주차장 종류명", "운영구분명",
    "주소", "총 주차면", "geometry"
]].rename(columns={
    "주차장코드": "id", 
    "주차장명": "name", 
    "주차장 종류명": "type",
    "운영구분명": "subtype", 
    "주소": "address", 
    "총 주차면": "capacity"
})

gdf_parking["facility"] = "주차장"

# 면적(총 주차면)을 기준으로 점수(idle_score) 계산
scaler = MinMaxScaler()
gdf_parking['idle_score'] = scaler.fit_transform(gdf_parking[['capacity']])

print(f"유효 좌표 확인: {gdf_parking.geometry.is_valid.sum()} / {len(gdf_parking)}")
display(gdf_parking.head(3))

주차장 원본: 2293개소
좌표 정상: 1531개소 | 좌표 누락: 762개소 (복구 시도)
🌐 누락된 주소 좌표 복구 중... (약간의 시간이 소요됩니다)
복구 성공: 93개소
최종 확보된 주차장 데이터: 1624개소
유효 좌표 확인: 1624 / 1624


,id,name,type,subtype,address,capacity,geometry,facility,idle_score
0,1037932,구로디지털단지역 공영주차장(시),노외 주차장,시간제 주차장,구로구 구로동 810-3,91,POINT (126.90124 37.48543),주차장,0.062937
1,1051043,구파발역 공영주차장(시),노외 주차장,시간제 주차장,은평구 진관동 70-1,399,POINT (126.91899 37.63821),주차장,0.278322
2,1163833,봉천복개3 공영주차장(시),노상 주차장,시간제 주차장,관악구 신림동 1467-3,1,POINT (126.93139 37.48807),주차장,0.000000


## 2. 통합 GeoDataFrame 저장

In [5]:
# 저장
gdf_all = gdf_parking.copy()
print(f"통합 공공시설: {len(gdf_all)}개소")

gdf_all.to_file(OUT / "public_facilities.gpkg", driver="GPKG")
print(f"\n저장 완료: {OUT / 'public_facilities.gpkg'}")

통합 공공시설: 1624개소

저장 완료: E:\서울시데이터경진대회\Aero-Logic-Seoul\processed\public_facilities.gpkg


In [8]:
import folium

# 성남시 중심 좌표
center = [37.5, 127.0]
m = folium.Map(location=center, zoom_start=12, tiles="cartodbpositron")

# 성남시 경계 오버레이
boundary = gpd.read_file(OUT / "seoul_boundary.gpkg", layer="city")
folium.GeoJson(boundary, style_function=lambda x: {
    "fillColor": "none", "color": "black", "weight": 2
}).add_to(m)

# 주차장 (파란 점)
for _, row in gdf_all[gdf_all["facility"] == "주차장"].iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3, color="blue", fill=True, fill_opacity=0.6,
        popup=f"{row['name']} (주차: {row['capacity']}면)",
    ).add_to(m)

# 복지센터 (빨간 점)
for _, row in gdf_all[gdf_all["facility"] == "행정복지센터"].iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color="red", fill=True, fill_opacity=0.8,
        popup=f"{row['name']}",
    ).add_to(m)

m